In [6]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pathlib import Path
import numpy as np

In [2]:
root_dir = Path(r'C:\python\powerpath\data')
assert root_dir.exists()
static_path = root_dir.joinpath("static")
hazard_path = static_path.joinpath(r"C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries")
hazard_reprojected_path = hazard_path.joinpath("reprojected") 
hazard_reprojected_path.mkdir(exist_ok=True)


In [3]:
def get_all_files(directory: Path) -> list[Path]:
    return [file for file in directory.iterdir() if file.is_file() and file.suffix == '.tif']

In [4]:
hazard_files = get_all_files(hazard_path)

for hazard_file in hazard_files:
    print (hazard_file)

C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-05_scale_5.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-06_scale_5.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-07_scale_5.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GLG_waterschijf_1900-07-0

In [7]:
def get_all_files(directory: Path) -> list[Path]:
    return [file for file in directory.iterdir() if file.is_file() and file.suffix == '.tif']

# Define target CRS
dst_crs = 'EPSG:4326'

# Get all hazard files
hazard_files = get_all_files(hazard_path)



for hazard_file in hazard_files:
    print(f"Reprojecting: {hazard_file.name}")
    
    dst_file = hazard_reprojected_path / f"{hazard_file.stem}_wgs84{hazard_file.suffix}"

    with rasterio.open(hazard_file) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': 0  # Set output nodata to 0
        })

        nodata = src.nodata

        with rasterio.open(dst_file, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reprojected = np.empty((height, width), dtype=src.dtypes[i - 1])

                reproject(
                    source=rasterio.band(src, i),
                    destination=reprojected,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest)

                # Replace NaNs, nodata, and negative values with 0
                reprojected = np.where(
                    np.isnan(reprojected) | 
                    (reprojected < 0) | 
                    ((nodata is not None) & (reprojected == nodata)),
                    0,
                    reprojected
                )

                dst.write(reprojected, i)



Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-05_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-06_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-07_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GLG_waterschijf_1900-07-08_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GLG_waterschijf_1900-07-09_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GLG_waterschijf_1900-07-10_scale_5.tif
Reprojecting: 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GLG_waterschijf_1900-07-11_scale_5.tif
